# **CS 584-A: Natural Language Processing**
# **Homework 4**

**Name: Ankit Dhandharia**

**CWID: 20033031**

## **Goals**
The goal of HW4 is for you to get hands-on experience of utilizing pre-trained and large language model for the machine reading comprehension task. The questions are open questions and there is no fixed solution. Different data selection and processing, parameter initialization, data split strategies will lead to different predictions and results. Therefore, during the grading, the specific values in the results are not required. It is important that you focus on implementing and setting up the pipelines of applying these models to solve the tasks.

## **Machine Reading Comprehension**

# **1. Data preparation**

## **1.1 Data download and preprocessing**

In [3]:
!pip install transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [4]:
from datasets import load_dataset
dataset = load_dataset('squad_v2')
dataset

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [5]:
selected_data = dataset['train'][:15]
selected_data

{'id': ['56be85543aeaaa14008c9063',
  '56be85543aeaaa14008c9065',
  '56be85543aeaaa14008c9066',
  '56bf6b0f3aeaaa14008c9601',
  '56bf6b0f3aeaaa14008c9602',
  '56bf6b0f3aeaaa14008c9603',
  '56bf6b0f3aeaaa14008c9604',
  '56bf6b0f3aeaaa14008c9605',
  '56d43c5f2ccc5a1400d830a9',
  '56d43c5f2ccc5a1400d830aa',
  '56d43c5f2ccc5a1400d830ab',
  '56d43c5f2ccc5a1400d830ac',
  '56d43c5f2ccc5a1400d830ad',
  '56d43ce42ccc5a1400d830b4',
  '56d43ce42ccc5a1400d830b5'],
 'title': ['Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé',
  'Beyoncé'],
 'context': ['Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Dest

## **1.2 Data statistics**

a) Statistics about the max/average/min length of the contexts

b) Statistics about the max/average/min length of the questions

c) Statistics about the max/average/min length of the answer

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Calculate statistics for context length
context_lengths = [len(context) for context in selected_data['context']]
context_stats = {
    'Category': 'Context',
    'Count': len(context_lengths),
    'Max Length': max(context_lengths),
    'Min Length': min(context_lengths),
    'Average Length': round(np.mean(context_lengths), 1)
}

# Calculate statistics for question length
question_lengths = [len(question) for question in selected_data["question"]]
question_stats = {
    'Category': 'Question',
    'Count': len(question_lengths),
    'Max Length': max(question_lengths),
    'Min Length': min(question_lengths),
    'Average Length': round(np.mean(question_lengths), 1)
}

# Calculate statistics for answer length
answer_lengths = [len(answer['text'][0]) for answer in selected_data['answers']]
answer_stats = {
    'Category': 'Answer',
    'Count': len(answer_lengths),
    'Max Length': max(answer_lengths),
    'Min Length': min(answer_lengths),
    'Average Length': round(np.mean(answer_lengths), 1)
}

# Combine statistics into a single table
stats_table = pd.DataFrame([context_stats, question_stats, answer_stats])

# Display the statistics table
stats_table


,Category,Count,Max Length,Min Length,Average Length
0,Context,15,694,694,694.0
1,Question,15,64,30,47.3
2,Answer,15,19,4,12.7


# **2. Question answering**

### **2.1 Pre-trained RoBERTa model to solve the question answering task on the data you prepared.**

In [9]:
from transformers import AutoTokenizer, RobertaForQuestionAnswering
import torch

# Load the pre-trained RoBERTa tokenizer and model for question answering
tokenizer = AutoTokenizer.from_pretrained(
    "deepset/roberta-base-squad2",
    truncation=True,
    max_length=512,
    padding="max_length"
)
model = RobertaForQuestionAnswering.from_pretrained("deepset/roberta-base-squad2")

# Perform question answering on the first 15 samples
for sample_idx in range(15):
    context_text = selected_data['context'][sample_idx]
    question_text = selected_data['question'][sample_idx]

    # Tokenize the question and context
    tokenized_inputs = tokenizer(question_text, context_text, return_tensors="pt")

    # Perform inference to get model outputs
    with torch.no_grad():
        outputs = model(**tokenized_inputs)

    # Identify start and end indices for the predicted answer
    start_position = torch.argmax(outputs.start_logits)
    end_position = torch.argmax(outputs.end_logits)

    # Extract token IDs for the predicted answer
    answer_token_ids = tokenized_inputs.input_ids[0, start_position : end_position + 1]

    # Decode the predicted answer tokens into text
    predicted_answer_text = tokenizer.decode(answer_token_ids, skip_special_tokens=True)

    # Display results for the current sample
    print(f"Sample #{sample_idx + 1}")
    print(f"Question: {question_text}")
    print(f"Start Index: {start_position}, End Index: {end_position}")
    print(f"Predicted Answer Token IDs: {answer_token_ids.tolist()}")
    print(f"Decoded Answer: {predicted_answer_text}")


Sample #1
Question: When did Beyonce start becoming popular?
Start Index: 87, End Index: 89
Predicted Answer Token IDs: [628, 4525, 29]
Decoded Answer:  late 1990s
Sample #2
Question: What areas did Beyonce compete in when she was growing up?
Start Index: 78, End Index: 80
Predicted Answer Token IDs: [6970, 8, 7950]
Decoded Answer:  singing and dancing
Sample #3
Question: When did Beyonce leave Destiny's Child and become a solo singer?
Start Index: 155, End Index: 155
Predicted Answer Token IDs: [35153]
Decoded Answer: 2003
Sample #4
Question: In what city and state did Beyonce  grow up? 
Start Index: 72, End Index: 72
Predicted Answer Token IDs: [1184]
Decoded Answer:  Texas
Sample #5
Question: In which decade did Beyonce become famous?
Start Index: 89, End Index: 90
Predicted Answer Token IDs: [4525, 29]
Decoded Answer:  1990s
Sample #6
Question: In what R&B group was she the lead singer?
Start Index: 104, End Index: 106
Predicted Answer Token IDs: [23313, 18, 7442]
Decoded Answer:  

In [10]:
# Adjust tokenizer settings
tokenizer = AutoTokenizer.from_pretrained(
    "deepset/roberta-base-squad2",
    truncation=False,  # Disable truncation
    max_length=128,    # Set maximum length
    padding=False      # Disable padding
)
model = RobertaForQuestionAnswering.from_pretrained("deepset/roberta-base-squad2")

# Perform question answering on the first 15 samples with updated tokenizer settings
for sample_idx in range(15):
    context_text = selected_data['context'][sample_idx]
    question_text = selected_data['question'][sample_idx]

    # Tokenize the question and context
    tokenized_inputs = tokenizer(question_text, context_text, return_tensors="pt")

    # Perform inference to get model outputs
    with torch.no_grad():
        outputs = model(**tokenized_inputs)

    # Identify start and end indices for the predicted answer
    start_position = torch.argmax(outputs.start_logits)
    end_position = torch.argmax(outputs.end_logits)

    # Extract token IDs for the predicted answer
    answer_token_ids = tokenized_inputs.input_ids[0, start_position : end_position + 1]

    # Decode the predicted answer tokens into text
    predicted_answer_text = tokenizer.decode(answer_token_ids, skip_special_tokens=True)

    # Display results for the current sample
    print(f"Sample #{sample_idx + 1}")
    print(f"Question: {question_text}")
    print(f"Start Index: {start_position}, End Index: {end_position}")
    print(f"Predicted Answer Token IDs: {answer_token_ids.tolist()}")
    print(f"Decoded Answer: {predicted_answer_text}")


Sample #1
Question: When did Beyonce start becoming popular?
Start Index: 87, End Index: 89
Predicted Answer Token IDs: [628, 4525, 29]
Decoded Answer:  late 1990s
Sample #2
Question: What areas did Beyonce compete in when she was growing up?
Start Index: 78, End Index: 80
Predicted Answer Token IDs: [6970, 8, 7950]
Decoded Answer:  singing and dancing
Sample #3
Question: When did Beyonce leave Destiny's Child and become a solo singer?
Start Index: 155, End Index: 155
Predicted Answer Token IDs: [35153]
Decoded Answer: 2003
Sample #4
Question: In what city and state did Beyonce  grow up? 
Start Index: 72, End Index: 72
Predicted Answer Token IDs: [1184]
Decoded Answer:  Texas
Sample #5
Question: In which decade did Beyonce become famous?
Start Index: 89, End Index: 90
Predicted Answer Token IDs: [4525, 29]
Decoded Answer:  1990s
Sample #6
Question: In what R&B group was she the lead singer?
Start Index: 104, End Index: 106
Predicted Answer Token IDs: [23313, 18, 7442]
Decoded Answer:  

## **Prompt:**

Provide the answers of the each question from the comprehension and reason behind the answer.

Beyoncé Giselle Knowles-Carter (born September 4, 1981) is an American singer, songwriter, record producer, and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child and rose to fame in the late 1990s as the lead singer of R&B girl group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world’s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé’s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards, and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".

### Questions:
1. When did Beyoncé start becoming popular?  
2. What areas did Beyoncé compete in when she was growing up?  
3. When did Beyoncé leave Destiny's Child and become a solo singer?  
4. In what city and state did Beyoncé grow up?  
5. In which decade did Beyoncé become famous?  
6. In what R&B group was she the lead singer?  
7. What album made her a worldwide known artist?  
8. Who managed the Destiny's Child group?  
9. When did Beyoncé rise to fame?  
10. What role did Beyoncé have in Destiny's Child?  
11. What was the first album Beyoncé released as a solo artist?  
12. When did Beyoncé release *Dangerously in Love*?  
13. How many Grammy awards did Beyoncé win for her first solo album?  
14. What was Beyoncé's role in Destiny's Child?  
15. What was the name of Beyoncé's first solo album?  


## **2.2 ChatGPT.**

Here are the answers to each question along with the reasoning:

1. **When did Beyoncé start becoming popular?**  
   **Answer**: Beyoncé started becoming popular in the late 1990s.  
   **Reason**: The text mentions she rose to fame in the late 1990s as the lead singer of Destiny's Child.

2. **What areas did Beyoncé compete in when she was growing up?**  
   **Answer**: Beyoncé competed in singing and dancing competitions.  
   **Reason**: The text states she performed in various singing and dancing competitions as a child.

3. **When did Beyoncé leave Destiny's Child and become a solo singer?**  
   **Answer**: Beyoncé became a solo singer after the hiatus of Destiny's Child.  
   **Reason**: The text notes that the group went on hiatus, during which Beyoncé released her debut solo album.

4. **In what city and state did Beyoncé grow up?**  
   **Answer**: Beyoncé grew up in Houston, Texas.  
   **Reason**: The text clearly states that she was born and raised in Houston, Texas.

5. **In which decade did Beyoncé become famous?**  
   **Answer**: Beyoncé became famous in the 1990s.  
   **Reason**: The text mentions that she rose to fame in the late 1990s.

6. **In what R&B group was she the lead singer?**  
   **Answer**: Beyoncé was the lead singer of Destiny's Child.  
   **Reason**: The text specifies that she was the lead singer of Destiny's Child.

7. **What album made her a worldwide known artist?**  
   **Answer**: Beyoncé became a worldwide known artist with her debut album *Dangerously in Love*.  
   **Reason**: The text indicates that this album established her as a solo artist worldwide.

8. **Who managed the Destiny's Child group?**  
   **Answer**: Mathew Knowles, Beyoncé’s father, managed Destiny's Child.  
   **Reason**: The text clearly states that the group was managed by her father, Mathew Knowles.

9. **When did Beyoncé rise to fame?**  
   **Answer**: Beyoncé rose to fame in the late 1990s.  
   **Reason**: The text specifically states that she rose to fame in the late 1990s as the lead singer of Destiny's Child.

10. **What role did Beyoncé have in Destiny's Child?**  
   **Answer**: Beyoncé was the lead singer of Destiny's Child.  
   **Reason**: The text directly states she was the lead singer of the group.

11. **What was the first album Beyoncé released as a solo artist?**  
   **Answer**: Beyoncé’s first album as a solo artist was *Dangerously in Love*.  
   **Reason**: The text mentions that *Dangerously in Love* was released after the group's hiatus.

12. **When did Beyoncé release *Dangerously in Love*?**  
   **Answer**: Beyoncé released *Dangerously in Love* in 2003.  
   **Reason**: The text specifically states that the album came out in 2003.

13. **How many Grammy awards did Beyoncé win for her first solo album?**  
   **Answer**: Beyoncé won five Grammy Awards for her first solo album.  
   **Reason**: The text clearly mentions that *Dangerously in Love* earned her five Grammy Awards.

14. **What was Beyoncé's role in Destiny's Child?**  
   **Answer**: Beyoncé was the lead singer of Destiny's Child.  
   **Reason**: This is confirmed by the text, which specifies her role in the group.

15. **What was the name of Beyoncé's first solo album?**  
   **Answer**: Beyoncé’s first solo album was *Dangerously in Love*.  
   **Reason**: The text names her debut solo album as *Dangerously in Love*.


## **2.3 Claude 3.5 Sonnet**

I'll provide answers to each question along with the reasoning from the text:

1. When did Beyoncé start becoming popular?
Answer: Late 1990s
Reasoning: The text states "rose to fame in the late 1990s as the lead singer of R&B girl group Destiny's Child"

2. What areas did Beyoncé compete in when she was growing up?
Answer: Singing and dancing competitions
Reasoning: Text mentions "performed in various singing and dancing competitions as a child"

3. When did Beyoncé leave Destiny's Child and become a solo artist?
Answer: 2003
Reasoning: The text mentions that her debut solo album was released in 2003 during the group's hiatus

4. In what city and state did Beyoncé grow up?
Answer: Houston, Texas
Reasoning: Text explicitly states "Born and raised in Houston, Texas"

5. In which decade did Beyoncé become famous?
Answer: 1990s
Reasoning: Text states she "rose to fame in the late 1990s"

6. In what R&B group was she the lead singer?
Answer: Destiny's Child
Reasoning: Text states "lead singer of R&B girl group Destiny's Child"

7. What album made her a worldwide known artist?
Answer: Dangerously in Love
Reasoning: Text mentions "Dangerously in Love (2003), which established her as a solo artist worldwide"

8. Who managed the Destiny's Child group?
Answer: Mathew Knowles (her father)
Reasoning: Text states "Managed by her father, Mathew Knowles"

9. When did Beyoncé rise to fame?
Answer: Late 1990s
Reasoning: Text explicitly states "rose to fame in the late 1990s"

10. What role did Beyoncé have in Destiny's Child?
Answer: Lead singer
Reasoning: Text states "lead singer of R&B girl group Destiny's Child"

11. What was the first album Beyoncé released as a solo artist?
Answer: Dangerously in Love
Reasoning: Text refers to it as "Beyoncé's debut album, Dangerously in Love"

12. When did Beyoncé release Dangerously in Love?
Answer: 2003
Reasoning: Text states "Dangerously in Love (2003)"

13. How many Grammy awards did Beyoncé win for her first solo album?
Answer: Five Grammy Awards
Reasoning: Text mentions that Dangerously in Love "earned five Grammy Awards"

14. What was Beyoncé's role in Destiny's Child?
Answer: Lead singer
Reasoning: Text states "lead singer of R&B girl group Destiny's Child"

15. What was the name of Beyoncé's first solo album?
Answer: Dangerously in Love
Reasoning: Text refers to it as "Beyoncé's debut album, Dangerously in Love"

# **3. Results analysis and evaluation**

### **3.1 Manual Checking of Output Answers.**

In [11]:
ground_truth = [item['text'][0] for item in selected_data['answers']]
ground_truth

['in the late 1990s',
 'singing and dancing',
 '2003',
 'Houston, Texas',
 'late 1990s',
 "Destiny's Child",
 'Dangerously in Love',
 'Mathew Knowles',
 'late 1990s',
 'lead singer',
 'Dangerously in Love',
 '2003',
 'five',
 'lead singer',
 'Dangerously in Love']

In [15]:
result_table_elements = {
    'Question': [
        'When did Beyoncé start becoming popular?',
        'What areas did Beyoncé compete in when she was growing up?',
        'When did Beyoncé leave Destiny\'s Child and become a solo singer?',
        'In what city and state did Beyoncé grow up?',
        'In which decade did Beyoncé become famous?',
        'In what R&B group was she the lead singer?',
        'What album made her a worldwide known artist?',
        'Who managed the Destiny\'s Child group?',
        'When did Beyoncé rise to fame?',
        'What role did Beyoncé have in Destiny\'s Child?',
        'What was the first album Beyoncé released as a solo artist?',
        'When did Beyoncé release Dangerously in Love?',
        'How many Grammy awards did Beyoncé win for her first solo album?',
        'What was Beyoncé\'s role in Destiny\'s Child?',
        'What was the name of Beyoncé\'s first solo album?'
    ],
    'Ground Truth': [
        'in the late 1990s',
        'singing and dancing',
        '2003',
        'Houston, Texas',
        'late 1990s',
        "Destiny's Child",
        'Dangerously in Love',
        'Mathew Knowles',
        'late 1990s',
        'lead singer',
        'Dangerously in Love',
        '2003',
        'five',
        'lead singer',
        'Dangerously in Love'
    ],
    'RoBERTa': [
        'late 1990s',
        'singing and dancing',
        '2003',
        'Texas',
        '1990s',
        "Destiny's Child",
        'Dangerously in Love',
        'Mathew Knowles',
        'late 1990s',
        'lead singer',
        'Dangerously in Love',
        '2003',
        'five',
        'lead singer',
        'Dangerously in Love'
    ],
    'ChatGPT': [
        'Beyoncé started becoming popular in the late 1990s.',
        'Singing and dancing competitions.',
        'Beyoncé became a solo singer after the hiatus of Destiny\'s Child.',
        'Houston, Texas.',
        'The 1990s.',
        'Destiny\'s Child.',
        'Dangerously in Love.',
        'Mathew Knowles.',
        'In the late 1990s.',
        'Lead singer.',
        'Dangerously in Love.',
        '2003.',
        'Five Grammy Awards.',
        'Lead singer.',
        'Dangerously in Love.'
    ],
    'Claude 3.5': [
        'Late 1990s',
        'Singing and dancing competitions',
        '2003',
        'Houston, Texas',
        '1990s',
        "Destiny's Child",
        'Dangerously in Love',
        'Mathew Knowles (her father)',
        'Late 1990s',
        'Lead singer',
        'Dangerously in Love',
        '2003',
        'Five Grammy Awards',
        'Lead singer',
        'Dangerously in Love'
    ]
}

# Create the result table as a pandas DataFrame
result_table = pd.DataFrame(result_table_elements)

# Display the result table
print(result_table)


                                                            Question  \
0                           When did Beyoncé start becoming popular?   
1         What areas did Beyoncé compete in when she was growing up?   
2   When did Beyoncé leave Destiny's Child and become a solo singer?   
3                        In what city and state did Beyoncé grow up?   
4                         In which decade did Beyoncé become famous?   
5                         In what R&B group was she the lead singer?   
6                      What album made her a worldwide known artist?   
7                             Who managed the Destiny's Child group?   
8                                     When did Beyoncé rise to fame?   
9                     What role did Beyoncé have in Destiny's Child?   
10       What was the first album Beyoncé released as a solo artist?   
11                     When did Beyoncé release Dangerously in Love?   
12  How many Grammy awards did Beyoncé win for her first solo al

#### General Score Calculation:

| Question | Ground Truth | RoBERTa | ChatGPT | Claude 3.5 | Correct Answer |
|----------|--------------|---------|---------|------------|----------------|
| All 15 questions | Correct answers provided | 15/15 | 15/15 | 15/15 | All correct |

All models achieved **15/15** correct answers.



### **3.2 Comparison and Discussion**

- **Performance**: All models performed equally well, scoring **15/15**.
- **Advantages**: Pre-trained models like **RoBERTa** are concise and accurate for factual questions. **ChatGPT** and **Claude 3.5** are more verbose but still accurate, often providing additional context.
- **Disadvantages**: **ChatGPT** and **Claude 3.5** can sometimes offer unnecessary elaboration. **RoBERTa** might miss finer details (e.g., just "Texas" instead of "Houston, Texas").



### **3.3 Specific Data Samples Comparison.**



- **Q1: "When did Beyoncé leave Destiny’s Child?"**
  - **RoBERTa/Claude 3.5**: 2003 (Exact)
  - **ChatGPT**: After the hiatus of Destiny's Child (Correct, but more detailed).

- **Q2: "In what city and state did Beyoncé grow up?"**
  - **RoBERTa**: Texas (Partially correct, lacks specificity)
  - **ChatGPT/Claude 3.5**: Houston, Texas (Exact).


# **Self-Reflection**


**Q1: Rate your performance**

Answer: Somewhat satisfied

**Q2: How much does your understanding of the concepts influence your performance?**

Answer: 4-Somewhat important

**Q3: How much does the design of the examples/questions (is relatable to you) influence your performance?**

Answer: Neutral

**Q4: When you answer Q3, which are the primary factors influencing the relatability of the examples to you?**

Answer: Prior knowledge about the context in the example